In [1]:
# ------------------------------------------------------------
# Step 1 check: confirm the environment has what we need
# ------------------------------------------------------------
import sys                 # tells us which Python is running
import numpy as np         # fast maths on arrays (the Markov trace)
import pandas as pd        # tables (results, sensitivity analyses)
import matplotlib          # charts (CE plane, tornado, CEAC)
import openpyxl            # writing the Excel model later

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("openpyxl:", openpyxl.__version__)

# This path should end in ...envs\health_econ\python.exe
# If it doesn't, Jupyter was started from the wrong environment.
print(sys.executable)

Python: 3.11.18
NumPy: 2.4.6
pandas: 3.0.6
matplotlib: 3.11.2
openpyxl: 3.1.5
C:\Users\huilu\.conda\envs\health_econ\python.exe


In [2]:
# ------------------------------------------------------------
# Model settings: the "rules of the game", not uncertain inputs
# ------------------------------------------------------------
import numpy as np
import pandas as pd

SETTINGS = {
    "start_age": 70,          # cohort starts treatment at 70 (CHADS2 = 2 profile)
    "max_age": 100,           # lifetime horizon: model runs until age 100
    "cycle_years": 0.25,      # 3-month cycles, same as the CADTH worked example
    "discount_rate": 0.015,   # 1.5%/yr on costs AND QALYs (CADTH 4th edition reference case)
    "wtp": 50_000,            # willingness-to-pay per QALY, used for net monetary benefit
}

# The worked example's costs are in 2011 dollars. We convert them to 2025
# dollars with the all-items CPI (Statistics Canada Table 18-10-0005-01).
CPI_2011 = 119.9
CPI_2025 = 164.2
INFLATE = CPI_2025 / CPI_2011

print(f"Inflator 2011 -> 2025: {INFLATE:.4f}")


Inflator 2011 -> 2025: 1.3695


In [5]:
# ------------------------------------------------------------
# How likely is a 70-, 80- or 90-year-old Canadian to die in a year?
# ------------------------------------------------------------
# qx = probability of dying within the year at that age.
# Source: Statistics Canada, complete life table, Canada, 2021-2023.
life_table = pd.DataFrame({
    "age": [70, 75, 80, 85, 90],
    "qx":  [0.01819, 0.02950, 0.04912, 0.08399, 0.14746],
})

# Mortality rises roughly exponentially with age (the Gompertz law).
# So log(qx) is close to a straight line in age, and fitting that line
# gives us a smooth qx for EVERY age from 70 to 100, not just these five.
slope, intercept = np.polyfit(life_table["age"], np.log(life_table["qx"]), 1)
print(f"Gompertz slope: {slope:.5f}   intercept: {intercept:.4f}")

# Check the fit: fitted values should sit close to the real ones
life_table["qx_fitted"] = np.exp(intercept + slope * life_table["age"])
print(life_table.round(5))

Gompertz slope: 0.10463   intercept: -11.3577
   age       qx  qx_fitted
0   70  0.01819    0.01772
1   75  0.02950    0.02989
2   80  0.04912    0.05044
3   85  0.08399    0.08511
4   90  0.14746    0.14361


In [6]:
# ------------------------------------------------------------
# Every uncertain parameter goes into one table.
# Columns:
#   base       value used in the main (deterministic) analysis
#   low, high  range for the one-way sensitivity analysis (tornado chart)
#   dist       distribution for the probabilistic analysis (Step 6)
#   a, b       distribution parameters:
#                beta      -> a = "effective sample size" (alpha + beta)
#                lognormal -> a, b = lower and upper 95% CI
#                gamma     -> a = coefficient of variation
#                normal    -> a = standard error
#   source     where the number comes from
# These distribution choices follow the CADTH worked example exactly.
# ------------------------------------------------------------
rows = []

def add(name, base, dist, a=None, b=None, low=None, high=None,
        source="CADTH WE", label=""):
    """Append one parameter to the input table."""
    rows.append(dict(name=name, label=label, base=base, low=low, high=high,
                     dist=dist, a=a, b=b, source=source))

# --- Annual event probabilities for a CHADS2 = 2 patient ON WARFARIN
#     (RE-LY trial warfarin arm, via the CADTH worked example, Table 1)
add("p_stroke",      0.016, "beta", a=186+11608, low=0.012,  high=0.020,  label="Stroke rate on warfarin (annual)")
add("p_tia",         0.008, "beta", a=99+11695,  low=0.006,  high=0.010,  label="TIA rate (annual)")
add("p_ich",         0.008, "beta", a=90+11704,  low=0.006,  high=0.010,  label="ICH rate on warfarin (annual)")
add("p_major_bleed", 0.033, "beta", a=386+11408, low=0.025,  high=0.041,  label="Major bleed rate on warfarin (annual)")
add("p_minor_bleed", 0.164, "beta", a=1931+9863, low=0.123,  high=0.205,  label="Minor bleed rate (annual)")
add("p_mi",          0.006, "beta", a=66+11728,  low=0.0045, high=0.0075, label="MI rate (annual)")

# --- What happens when an event occurs
add("f_stroke_fatal", 0.237, "beta", a=44+142, low=0.18, high=0.30, label="Proportion of first strokes fatal")
add("f_stroke_major", 0.333, "beta", a=39+78,  low=0.25, high=0.42, label="Proportion of non-fatal strokes major")
add("hr_recur_fatal", 1.57, "lognormal", a=1.21, b=2.03, low=1.21, high=2.03, label="Increase in fatality of recurrent stroke")
add("f_bleed_fatal",  0.084, "beta", a=40+436, low=0.06, high=0.11, label="Proportion of major bleeds / ICH fatal")
add("f_mi_fatal",     0.121, "beta", a=8+58,   low=0.06, high=0.20, label="Proportion of MIs fatal")

# --- Risk modifiers
add("rr_stroke_per_decade",  1.50, "lognormal", a=1.30, b=1.70, low=1.30, high=1.70, label="Stroke risk per 10 years of age")
add("rr_stroke_prior",       2.20, "lognormal", a=0.78, b=6.35, low=1.20, high=3.50, label="Stroke risk after prior stroke/ICH")
add("hr_death_prior_stroke", 2.30, "lognormal", a=2.00, b=2.70, low=2.00, high=2.70, label="Mortality after stroke")
add("p_nonvasc_death",       0.033, "beta", a=391+11403, low=0.025, high=0.041, label="Non-vascular death at age 70 (annual)")

# --- THE KEY INPUTS: apixaban vs warfarin, odds ratios (< 1 = apixaban better)
#     From CADTH's network meta-analysis. Cross-check with the ARISTOTLE trial
#     (NEJM 2011): stroke HR 0.79, major bleeding HR 0.69. Close agreement.
add("or_apx_stroke", 0.79, "lognormal", a=0.65, b=0.95, low=0.65, high=0.95, label="Apixaban OR: stroke")
add("or_apx_mi",     0.88, "lognormal", a=0.66, b=1.17, low=0.66, high=1.17, label="Apixaban OR: MI")
add("or_apx_ich",    0.42, "lognormal", a=0.30, b=0.58, low=0.30, high=0.58, label="Apixaban OR: ICH")
add("or_apx_mb",     0.80, "lognormal", a=0.68, b=0.94, low=0.68, high=0.94, label="Apixaban OR: major bleed")

# --- After a stroke, ICH or major bleed, patients stop anticoagulation and
#     switch to aspirin (CADTH assumption). Aspirin's risks relative to warfarin:
add("rr_asa_stroke", 1.62, "lognormal", a=0.99, b=2.65, low=0.99, high=2.65, label="Aspirin RR: stroke")
add("rr_asa_mi",     1.42, "lognormal", a=0.84, b=2.39, low=0.84, high=2.39, label="Aspirin RR: MI")
add("rr_asa_ich",    0.51, "lognormal", a=0.16, b=1.60, low=0.16, high=1.60, label="Aspirin RR: ICH")
add("rr_asa_minor",  0.63, "lognormal", a=0.32, b=1.22, low=0.32, high=1.22, label="Aspirin RR: minor bleed")
add("rr_asa_mb",     1.14, "lognormal", a=0.47, b=2.73, low=0.47, high=2.73, label="Aspirin RR: major bleed")
add("rr_asa_tia",    1.56, "lognormal", a=0.86, b=2.83, low=0.86, high=2.83, label="Aspirin RR: TIA")

# --- Utilities (quality of life, 0 = dead, 1 = full health)
add("u_af",           0.810, "beta", a=33.82+7.93,   low=0.75, high=0.87, label="Utility: AF, no prior event")
add("u_minor_stroke", 0.750, "beta", a=86.69+28.90,  low=0.67, high=0.83, label="Utility: after minor stroke / ICH")
add("u_rankin_3_4",   0.390, "beta", a=69.74+109.08, low=0.32, high=0.46, label="Utility: major stroke, Rankin 3-4")
add("u_rankin_5",     0.110, "beta", a=18.93+153.16, low=0.06, high=0.16, label="Utility: major stroke, Rankin 5")
add("p_rankin_5",     0.205, "beta", a=8+39,         low=0.10, high=0.31, label="Share of major strokes at Rankin 5")
add("dec_per_year_over_70", 0.00029, "normal", a=0.00002, low=0.00025, high=0.00033, label="Utility loss per year over 70")
# One-off QALY losses, applied only in the 3-month cycle the event happens
add("dec_mi",          0.125, "normal", a=0.009, low=0.107, high=0.143, label="Disutility: MI")
add("dec_major_bleed", 0.092, "normal", a=0.010, low=0.072, high=0.112, label="Disutility: major bleed")
add("dec_minor_bleed", 0.013, "normal", a=0.001, low=0.011, high=0.015, label="Disutility: minor bleed")
add("dec_tia",         0.103, "normal", a=0.008, low=0.087, high=0.119, label="Disutility: TIA")

print("Clinical parameters entered:", len(rows))

Clinical parameters entered: 35


In [7]:
# ------------------------------------------------------------
# Event and long-term costs: entered in 2011 CAD, converted to 2025 CAD
# ------------------------------------------------------------
costs_2011 = {
    # name:               (2011 CAD,  label)
    "c_stroke_fatal":     (16_800, "Cost: fatal stroke"),
    "c_stroke_minor":     (16_800, "Cost: minor stroke"),
    "c_stroke_major":     (56_864, "Cost: major stroke (acute)"),
    "c_tia":              (4_296,  "Cost: TIA"),
    "c_ich":              (16_559, "Cost: ICH"),
    "c_major_bleed":      (4_392,  "Cost: major bleed"),
    "c_minor_bleed":      (104,    "Cost: minor bleed"),
    "c_mi_fatal":         (7_351,  "Cost: fatal MI"),
    "c_mi_nonfatal":      (11_380, "Cost: non-fatal MI"),
    # Ongoing annual costs of living with the consequences of a stroke
    "c_lt_major_stroke":  (19_069, "Annual cost: after major stroke"),
    "c_lt_minor_stroke":  (7_896,  "Annual cost: after minor stroke / ICH"),
}
for name, (v2011, label) in costs_2011.items():
    v = v2011 * INFLATE
    # Gamma with a 25% coefficient of variation: the worked example's convention
    # when a study reports no measure of spread. Tornado range = +/-25%.
    add(name, v, "gamma", a=0.25, low=0.75 * v, high=1.25 * v,
        source="CADTH WE x CPI", label=label)

# ------------------------------------------------------------
# Drug costs: TODAY's Ontario Drug Benefit (ODB) prices. NOT inflated,
# because generic prices fall over time rather than tracking CPI.
# ------------------------------------------------------------
APX_GENERIC_TAB = 0.4084   # generic apixaban 5 mg, per tablet (ODB, Oct 2026)
APX_BRAND_TAB   = 1.6336   # Eliquis 5 mg brand, per tablet (ODB, Oct 2026)
DAYS = 365.25

c_apx = APX_GENERIC_TAB * 2 * DAYS            # 5 mg twice daily
add("c_apixaban", c_apx, "fixed", low=0.5 * c_apx, high=APX_BRAND_TAB * 2 * DAYS,
    source="ODB formulary, Oct 2026", label="Apixaban 5 mg BID, annual")
# NOTE: the high end of the tornado range is the BRAND price, so the chart
# directly answers "what if generics hadn't arrived?"

add("c_warfarin", 54.61, "fixed", low=27.0, high=110.0,
    source="CADTH WE (2011 ODB)", label="Warfarin 5 mg daily, annual")
c_inr = 240.69 * INFLATE                      # INR blood-test monitoring for warfarin
add("c_inr", c_inr, "fixed", low=0.5 * c_inr, high=1.5 * c_inr,
    source="CADTH WE x CPI", label="INR monitoring, annual (warfarin only)")
add("c_aspirin", 39.04, "fixed", low=20.0, high=60.0,
    source="CADTH WE (2011 ODB)", label="Aspirin, annual")

# Turn the list of rows into a table indexed by parameter name
inputs = pd.DataFrame(rows).set_index("name")

print("Number of parameters:", len(inputs))
print(inputs.loc[["c_apixaban", "c_inr", "c_stroke_major", "c_lt_major_stroke"],
                 ["base", "low", "high"]].round(2))
print(inputs["dist"].value_counts())

Number of parameters: 50
                       base       low      high
name                                           
c_apixaban           298.34    149.17   1193.34
c_inr                329.62    164.81    494.43
c_stroke_major     77873.80  58405.35  97342.25
c_lt_major_stroke  26114.51  19585.88  32643.14
dist
beta         16
lognormal    14
gamma        11
normal        5
fixed         4
Name: count, dtype: int64


In [8]:
# ------------------------------------------------------------
# Save inputs so Step 3 onwards (and the Excel model) read the SAME numbers
# ------------------------------------------------------------
import json

inputs.to_csv("../outputs/inputs.csv")
life_table.to_csv("../outputs/life_table.csv", index=False)

# Settings and the Gompertz fit go in a small JSON file
with open("../outputs/settings.json", "w") as f:
    json.dump({**SETTINGS,
               "gompertz_slope": slope,
               "gompertz_intercept": intercept,
               "inflate_2011_to_2025": INFLATE}, f, indent=2)

print("Saved: inputs.csv, life_table.csv, settings.json")

Saved: inputs.csv, life_table.csv, settings.json
